# BaTiO₃: training, polarization, BECs, and finite-field MD

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/01_batio3_workflow.ipynb)

This standalone workflow trains a compact MACEField model on a deterministic subset of the QE-labeled BaTiO₃ response data, then demonstrates Ti-displacement response and a longer alternating-field molecular-dynamics sweep. The 135-atom configurations came from Flare active learning, not AIMD. The bundled QE labels are coherent zero-field energy, force, stress, polarization, BEC, and polarizability labels on the same structures.

The training subset is a short teaching example. The single-point and MD sections use the bundled specialist checkpoint, so their results do not depend on running or saving the training cell first. The MD sweep reaches ±0.36 V/Å, 100 times the largest field magnitude in the bundled response labels (±0.0036 V/Å). Treat its P–E loop as an extrapolation stress test, not a validated physical hysteresis prediction.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["PYTHONWARNINGS"] = "ignore::DeprecationWarning,ignore::FutureWarning"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def ensure_checkout(path, url):
    path = Path(path)
    if (path / ".git").is_dir():
        branch = subprocess.run(
            ["git", "-C", str(path), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch == "main":
            return
        subprocess.run(["git", "-C", str(path), "fetch", "origin", "main"], check=True)
        subprocess.run(
            ["git", "-C", str(path), "checkout", "-B", "main", "origin/main"],
            check=True,
        )
        return
    if path.exists() and any(path.iterdir()):
        raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", url, str(path)],
        check=True,
    )


if IN_COLAB:
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    PACKAGE_REPO = Path("/content/mace-field")
    ensure_checkout(TUTORIAL_REPO, TUTORIAL_URL)
    ensure_checkout(PACKAGE_REPO, PACKAGE_URL)
else:
    CWD = Path.cwd().resolve()
    TUTORIAL_REPO = next(
        (
            path
            for path in (CWD, *CWD.parents)
            if (path / "tutorials/macefield/data").is_dir()
            and (path / "README.md").is_file()
        ),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run this notebook from a clone of mdi-group/mace-field-tutorial, "
            "or open it in Colab."
        )

    requested_repo = os.environ.get("MACEFIELD_REPO")
    if requested_repo:
        PACKAGE_REPO = Path(requested_repo).expanduser().resolve()
        if not (PACKAGE_REPO / "mace/modules/extensions.py").is_file():
            raise RuntimeError("MACEFIELD_REPO does not point to a MACE-Field checkout.")
        branch = subprocess.run(
            ["git", "-C", str(PACKAGE_REPO), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch != "main":
            raise RuntimeError("Set MACEFIELD_REPO to a checkout on the main branch.")
    else:
        candidates = [
            TUTORIAL_REPO.parent / "mace-field-main",
            TUTORIAL_REPO.parent / "mace-field",
        ]
        PACKAGE_REPO = None
        for candidate in candidates:
            if not (candidate / "mace/modules/extensions.py").is_file():
                continue
            branch = subprocess.run(
                ["git", "-C", str(candidate), "branch", "--show-current"],
                capture_output=True,
                text=True,
                check=False,
            ).stdout.strip()
            if branch == "main":
                PACKAGE_REPO = candidate.resolve()
                break
        if PACKAGE_REPO is None:
            PACKAGE_REPO = Path("/tmp/mace-field-main")
            ensure_checkout(PACKAGE_REPO, PACKAGE_URL)

os.environ["MACEFIELD_REPO"] = str(PACKAGE_REPO)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, str(PACKAGE_REPO))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--editable", str(PACKAGE_REPO)],
    check=True,
)
os.chdir(TUTORIAL_REPO)

TUTORIAL_DIR = TUTORIAL_REPO / "tutorials/macefield"
DATA_DIR = TUTORIAL_DIR / "data"
print("Tutorial repository:", TUTORIAL_REPO)
print("MACE-Field source:", PACKAGE_REPO)


def run_streamed(command, *, cwd):
    """Stream a command's stdout and stderr into the notebook cell."""
    child_env = os.environ.copy()
    child_env["PYTHONUNBUFFERED"] = "1"
    child_env["PYTHONWARNINGS"] = "ignore::DeprecationWarning,ignore::FutureWarning"
    print("Running command:", command, flush=True)
    process = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=child_env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


## 1. Inspect the training subset

The 12/4 subset is sampled across the fixed training and validation split. This section checks the cell sizes, available labels, and zero-field tags before fitting.


In [ ]:
import numpy as np
from ase.io import read

TRAIN = DATA_DIR / "batio3/batio3_qe_train_sample.extxyz"
VALID = DATA_DIR / "batio3/batio3_qe_valid_sample.extxyz"
train = read(TRAIN, index=":")
valid = read(VALID, index=":")
print("train / validation frames:", len(train), "/", len(valid))
print("atoms per frame:", sorted({len(a) for a in train + valid}))
print("info label keys:", sorted(train[0].info))
print("array label keys:", sorted(k for k in train[0].arrays if k.startswith("REF_")))
print("electric fields:", np.unique(
    np.round([a.info["REF_electric_field"] for a in train + valid], 8), axis=0
))


## Train a compact response model

Each epoch streams the validation loss and energy, force, stress, polarization, BEC, and polarizability RMSEs into the cell output. The default run is intentionally short for Colab. Increase `MACEFIELD_TUTORIAL_EPOCHS` for a longer demonstration. The small subset is for data and configuration practice, not for establishing converged accuracy.


In [ ]:
import os
import time
import yaml
import torch
import subprocess
import sys

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEFAULT_EPOCHS = 5 if DEVICE == "cuda" else 1
EPOCHS = int(os.environ.get("MACEFIELD_TUTORIAL_EPOCHS", DEFAULT_EPOCHS))
RUN_DIR = TUTORIAL_DIR / "runs" / f"batio3_qe_sample_{time.strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

config = {
    "name": "MACEField-BaTiO3-QE-sample",
    "model": "MACEField",
    "loss": "universal_field",
    "train_file": str(TRAIN),
    "valid_file": str(VALID),
    "valid_fraction": 0,
    "E0s": "average",
    "energy_key": "REF_energy",
    "forces_key": "REF_forces",
    "stress_key": "REF_stress",
    "electric_field_key": "REF_electric_field",
    "polarization_key": "REF_polarization",
    "becs_key": "REF_becs",
    "polarizability_key": "REF_polarizability",
    "compute_forces": True,
    "compute_stress": True,
    "compute_polarization": True,
    "compute_becs": True,
    "compute_polarizability": True,
    "energy_weight": 1.0,
    "forces_weight": 50.0,
    "stress_weight": 1.0,
    "polarization_weight": 10.0,
    "becs_weight": 50.0,
    "polarizability_weight": 10.0,
    "hidden_irreps": "16x0e + 16x1o",
    "num_interactions": 2,
    "max_L": 1,
    "r_max": 5.0,
    "batch_size": 1,
    "valid_batch_size": 1,
    "num_workers": 0,
    "lr": 0.005,
    "max_num_epochs": EPOCHS,
    "patience": max(3, EPOCHS),
    "eval_interval": 1,
    "error_table": "PerAtomFieldRMSE",
    "seed": 23,
    "default_dtype": "float32",
    "device": DEVICE,
    "model_dir": str(RUN_DIR / "models"),
    "checkpoints_dir": str(RUN_DIR / "checkpoints"),
    "log_dir": str(RUN_DIR / "logs"),
    "results_dir": str(RUN_DIR / "results"),
    "work_dir": str(RUN_DIR / "work"),
}
for output_key in ("model_dir", "checkpoints_dir", "log_dir", "results_dir", "work_dir"):
    Path(config[output_key]).mkdir(parents=True, exist_ok=True)
CONFIG = RUN_DIR / "train.yml"
CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print("device:", DEVICE, "| epochs:", EPOCHS)
print("configuration:", CONFIG)
print("output model directory:", config["model_dir"])

print(
    "Streaming per-epoch validation loss and energy, force, stress, polarization, "
    "BEC, and polarizability RMSEs (Hessian RMSE is None for this dataset).",
    flush=True,
)
run_streamed(
    [sys.executable, "-m", "mace.cli.run_train", "--config", str(CONFIG)],
    cwd=PACKAGE_REPO,
)


## What this small fit can show

The model receives zero-field energy, force, stress, polarization, BEC, and polarizability supervision. The subsample is deliberately small and has no field reversal or switching-path labels. Use it to learn the MACEField data keys and response losses, not to report quantitative BaTiO₃ or hysteresis predictions.

## 2. Shift Ti and inspect polarization and Born effective charges

This section evaluates configurations around a selected Ti displacement and compares the predicted BEC with a finite-difference derivative of polarization. It loads the bundled BTO checkpoint directly.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read
from mace.calculators import MACECalculator

MODEL = DATA_DIR / "batio3/MACE-Field-BaTiO3.model"
SAMPLE = DATA_DIR / "macefield_batio3_reference.extxyz"
source = read(SAMPLE, index=0)
volume = source.get_volume()

symbols = np.asarray(source.get_chemical_symbols())
ti_candidates = np.flatnonzero(symbols == "Ti")
fractional = source.get_scaled_positions(wrap=True)
center_delta = fractional[ti_candidates] - 0.5
center_delta -= np.rint(center_delta)
ti_index = int(ti_candidates[np.argmin(np.linalg.norm(center_delta, axis=1))])
source.info.clear()
for key in list(source.arrays):
    if key not in {"numbers", "positions"}:
        source.arrays.pop(key)


def structure_at(displacement):
    atoms = source.copy()
    atoms.positions[ti_index, 2] += float(displacement)
    atoms.info.clear()
    return atoms


device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print("Loading the bundled BaTiO3 response checkpoint...", flush=True)
calc = MACECalculator(
    model_paths=str(MODEL),
    model_type="MACEField",
    device=device,
    default_dtype="float32",
    electric_field=[0.0, 0.0, 0.0],
    compute_polarization=True,
    compute_becs=True,
    compute_polarizability=False,
)

displacements = np.linspace(-0.12, 0.12, 7)
fields = [-0.0036, 0.0, 0.0036]
polarizations = {field: [] for field in fields}
ti_zz = []

print(f"Evaluating {len(displacements)} Ti displacements and {len(fields)} fields.", flush=True)
for displacement_index, displacement in enumerate(displacements, start=1):
    print(f"Ti displacement {displacement_index}/{len(displacements)}: {displacement:+.4f} A", flush=True)
    atoms = structure_at(displacement)
    calc.electric_field = [0.0, 0.0, 0.0]
    calc.reset()
    atoms.calc = calc
    atoms.get_potential_energy()
    becs = np.asarray(calc.results["becs"]).reshape(len(atoms), 3, 3)
    ti_zz.append(float(becs[ti_index, 2, 2]))
    print(f"  zero-field BEC complete: Z*zz={ti_zz[-1]:.4f} e", flush=True)

    for field_index, field_z in enumerate(fields, start=1):
        calc.electric_field = [0.0, 0.0, float(field_z)]
        calc.reset()
        atoms.calc = calc
        atoms.get_potential_energy()
        pz = float(calc.results["polarization"][2])
        polarizations[field_z].append(pz)
        print(f"  field {field_index}/{len(fields)} | Ez={field_z:+.4f} V/A | Pz={pz:+.6e} e/A^2", flush=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for field_z in fields:
    axes[0].plot(displacements, polarizations[field_z], "o-", label=f"Ez={field_z:+.4f} V/Å")
axes[0].set(xlabel="selected Ti displacement along z (Å)", ylabel="Pz (e/Å²)",
            title="Field-dependent polarization")
axes[0].legend()
axes[1].plot(displacements, ti_zz, "o-", color="#9b4f96")
axes[1].set(xlabel="selected Ti displacement along z (Å)", ylabel="Ti Z*zz (e)",
            title="Predicted Born effective charge")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


## Check the BEC against a finite difference

For a cell polarization density, the Born charge includes the cell volume: $Z^*_{zz}=\Omega\,\partial P_z/\partial r_{\mathrm{Ti},z}$. This checks the model derivative against a central finite difference at zero field.


In [ ]:
delta = 1.0e-3
d0 = 0.04
field_z = 0.0

print("Evaluating zero-field model BEC at the finite-difference center...", flush=True)
atoms0 = structure_at(d0)
calc.electric_field = [0.0, 0.0, field_z]
calc.reset()
atoms0.calc = calc
atoms0.get_potential_energy()
z_model = float(
    np.asarray(calc.results["becs"]).reshape(len(atoms0), 3, 3)[ti_index, 2, 2]
)

print(f"  model Z*zz={z_model:.6f} e", flush=True)
p_values = []
for fd_index, displacement in enumerate((d0 - delta, d0 + delta), start=1):
    print(f"Finite-difference polarization sample {fd_index}/2 at displacement {displacement:+.4f} A...", flush=True)
    atoms = structure_at(displacement)
    calc.electric_field = [0.0, 0.0, field_z]
    calc.reset()
    atoms.calc = calc
    atoms.get_potential_energy()
    p_values.append(float(calc.results["polarization"][2]))
    print(f"  Pz={p_values[-1]:+.6e} e/A^2", flush=True)

z_fd = volume * (p_values[1] - p_values[0]) / (2 * delta)
print("model Ti Z*zz (e):", z_model)
print("finite-difference Ti Z*zz (e):", z_fd)


## 3. Sweep a broad electric field during MD

Every MD step prints time, target and instantaneous temperature, field, energy, and polarization. The notebook saves ASE trajectories, plots every per-step value, and animates sampled positions across each full trajectory. It sweeps from −0.36 to +0.36 V/Å and back at 25 K and 200 K. These fields are far beyond the response-label range, so any apparent switching or loop opening is exploratory model extrapolation, not a validated coercive field or hysteresis prediction.


## Set up a two-temperature field cycle

The default field cycle uses 13 evenly spaced values from −0.36 to +0.36 V/Å, then returns to −0.36 V/Å. It spends 100 MD steps (50 fs) at each field value. This is 100 times the largest reference field magnitude; use it to inspect the model response and apparent loop shape, not as a physical switching protocol. Set `MACEFIELD_BTO_MAX_FIELD_V_PER_A` or `MACEFIELD_BTO_STEPS_PER_FIELD` to change the range or dwell time. The longer run has 2,500 steps per temperature and can be shortened through the dwell-time variable on slower hardware.


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from ase import units
from ase.io import read, Trajectory
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from mace.calculators import MACECalculator

MODEL = DATA_DIR / "batio3/MACE-Field-BaTiO3.model"
SAMPLE = DATA_DIR / "macefield_batio3_reference.extxyz"
source = read(SAMPLE, index=0)
source.info.clear()
for key in list(source.arrays):
    if key not in {"numbers", "positions"}:
        source.arrays.pop(key)

max_field_V_per_A = float(os.environ.get("MACEFIELD_BTO_MAX_FIELD_V_PER_A", "0.36"))
if max_field_V_per_A <= 0:
    raise ValueError("MACEFIELD_BTO_MAX_FIELD_V_PER_A must be positive")
field_up = np.linspace(-max_field_V_per_A, max_field_V_per_A, 13)
field_cycle = np.concatenate([field_up, field_up[-2::-1]])
temperatures_K = [25.0, 200.0]
steps_per_field = int(os.environ.get("MACEFIELD_BTO_STEPS_PER_FIELD", "100"))
if steps_per_field < 1:
    raise ValueError("MACEFIELD_BTO_STEPS_PER_FIELD must be at least 1")
dt_fs = 0.5
friction_per_fs = 0.02
device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
MD_DIR = TUTORIAL_DIR / "runs" / f"batio3_field_md_{time.strftime('%Y%m%d_%H%M%S')}"
MD_DIR.mkdir(parents=True, exist_ok=False)
print("Writing step-by-step MD trajectories under:", MD_DIR, flush=True)
print(
    f"Field range: {field_cycle.min():+.3f} to {field_cycle.max():+.3f} V/Å "
    f"({len(field_up)} levels per sweep direction); "
    f"{steps_per_field} steps per field level.", flush=True
)

records_by_temperature = {}
for temperature_K in temperatures_K:
    atoms = source.copy()
    rng = np.random.default_rng(23 + int(temperature_K))
    MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K, rng=rng)
    Stationary(atoms)

    calc = MACECalculator(
        model_paths=str(MODEL),
        model_type="MACEField",
        device=device,
        default_dtype="float32",
        electric_field=[0.0, 0.0, float(field_cycle[0])],
        compute_polarization=True,
        compute_becs=False,
        compute_polarizability=False,
    )
    atoms.calc = calc
    dyn = Langevin(
        atoms,
        timestep=dt_fs * units.fs,
        temperature_K=temperature_K,
        friction=friction_per_fs / units.fs,
        rng=rng,
    )

    trajectory_path = MD_DIR / f"batio3_{int(temperature_K)}K.traj"
    trajectory = Trajectory(str(trajectory_path), "w", atoms)
    polarization_z = []
    energy = []
    trajectory_energy = []
    trajectory_polarization_z = []
    trajectory_temperature = []
    trajectory_fields_z = []
    trajectory_positions = []
    total_steps = len(field_cycle) * steps_per_field
    md_step = 0
    print(
        f"Starting {temperature_K:g} K run: {total_steps} steps "
        f"({total_steps * dt_fs / 1000:.3f} ps), dt={dt_fs:g} fs, "
        f"alternating field cycle.", flush=True
    )

    for field_index, field_z in enumerate(field_cycle, start=1):
        calc.electric_field = [0.0, 0.0, float(field_z)]
        calc.reset()
        segment_energy = []
        segment_polarization = []
        print(
            f"Field segment {field_index}/{len(field_cycle)}: "
            f"Ez={field_z:+.5f} V/Å", flush=True
        )
        for _ in range(steps_per_field):
            dyn.run(1)
            md_step += 1
            energy_now = float(atoms.get_potential_energy())
            polarization_now = float(calc.results["polarization"][2])
            temperature_now = float(atoms.get_temperature())
            trajectory.write(atoms)
            trajectory_energy.append(energy_now)
            trajectory_polarization_z.append(polarization_now)
            trajectory_temperature.append(temperature_now)
            trajectory_fields_z.append(float(field_z))
            trajectory_positions.append(atoms.positions.copy())
            segment_energy.append(energy_now)
            segment_polarization.append(polarization_now)
            print(
                f"MD step {md_step:04d}/{total_steps} | t={md_step * dt_fs:6.1f} fs "
                f"| Ttarget={temperature_K:5.1f} K | Tinst={temperature_now:7.2f} K "
                f"| Ez={field_z:+.5f} V/Å | E={energy_now:.6f} eV "
                f"| Pz={polarization_now:+.6e} e/Å²",
                flush=True,
            )
        energy.append(segment_energy[-1])
        polarization_z.append(segment_polarization[-1])

    trajectory.close()
    records_by_temperature[temperature_K] = {
        "field_z": field_cycle.copy(),
        "polarization_z": np.asarray(polarization_z),
        "energy": np.asarray(energy),
        "trajectory_energy": np.asarray(trajectory_energy),
        "trajectory_polarization_z": np.asarray(trajectory_polarization_z),
        "trajectory_temperature": np.asarray(trajectory_temperature),
        "trajectory_fields_z": np.asarray(trajectory_fields_z),
        "trajectory_positions": np.asarray(trajectory_positions),
        "forward_steps": len(field_up) * steps_per_field,
        "cell": np.asarray(source.cell.array).copy(),
        "symbols": np.asarray(source.get_chemical_symbols()),
        "trajectory_path": str(trajectory_path),
    }
    print(
        f"Finished {temperature_K:g} K run; saved {md_step} frames to {trajectory_path}",
        flush=True,
    )


## Plot polarization during the field cycle

Polarization is projected on z and converted to microcoulombs per square centimetre. The increasing and decreasing branches are plotted separately for readability. Any separation is an exploratory finite-rate response of a model evaluated far outside its field-label range; it does not establish a physical coercive field or validated ferroelectric hysteresis.


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

P_TO_UC_CM2 = 1602.176634
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for temperature_K, data in records_by_temperature.items():
    t_fs = np.arange(len(data["trajectory_energy"])) * dt_fs
    p = data["trajectory_polarization_z"] * P_TO_UC_CM2
    label = f"{temperature_K:g} K"
    forward_steps = data["forward_steps"]
    line = axes[0, 0].plot(
        data["trajectory_fields_z"][:forward_steps], p[:forward_steps],
        "-", linewidth=1.5, label=f"{label} increasing",
    )[0]
    axes[0, 0].plot(
        data["trajectory_fields_z"][forward_steps:], p[forward_steps:],
        "--", linewidth=1.5, color=line.get_color(), label=f"{label} decreasing",
    )
    axes[0, 1].plot(t_fs, p, label=label)
    axes[1, 0].plot(t_fs, data["trajectory_temperature"], label=label)
    axes[1, 1].plot(t_fs, data["trajectory_energy"], label=label)
axes[0, 0].set(xlabel="Ez (V/Å)", ylabel="Pz (µC/cm²)", title="Predicted P–E cycle (solid: increasing, dashed: decreasing)")
axes[0, 1].set(xlabel="time (fs)", ylabel="Pz (µC/cm²)", title="Polarization trajectory")
axes[1, 0].set(xlabel="time (fs)", ylabel="instantaneous T (K)", title="Thermostat trajectory")
axes[1, 1].set(xlabel="time (fs)", ylabel="energy (eV)", title="Potential-energy trajectory")
for ax in axes.flat:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

# Animate the saved atom positions at both temperatures, wrapping them into the cell.
records = list(records_by_temperature.items())
frame_count = min(len(data["trajectory_positions"]) for _, data in records)
# Sample at most 100 frames across each full trajectory to keep the Colab animation compact.
frame_ids = np.unique(np.linspace(0, frame_count - 1, min(frame_count, 100), dtype=int))
fig = plt.figure(figsize=(10, 4.5))
axes_3d = [fig.add_subplot(1, 2, j + 1, projection="3d") for j in range(2)]
colors = {"Ba": "#8c62aa", "Ti": "#4c78a8", "O": "#e45756"}

def wrapped_positions(data, frame):
    cell = data["cell"]
    frac = data["trajectory_positions"][frame] @ np.linalg.inv(cell)
    frac -= np.floor(frac)
    return frac @ cell

def draw_cell(ax, cell):
    corners = np.asarray([
        np.asarray(bits, dtype=float) @ cell
        for bits in ((0,0,0),(0,0,1),(0,1,0),(0,1,1),(1,0,0),(1,0,1),(1,1,0),(1,1,1))
    ])
    for idx, start in enumerate(corners):
        for axis in range(3):
            if not (idx & (1 << axis)):
                end = start + cell[axis]
                ax.plot(*np.vstack([start, end]).T, color="0.55", linewidth=0.7)
    limits = np.column_stack([corners.min(axis=0), corners.max(axis=0)])
    ax.set_xlim(*limits[0]); ax.set_ylim(*limits[1]); ax.set_zlim(*limits[2])
    ax.set_box_aspect(np.maximum(np.ptp(corners, axis=0), 1.0e-6))

def update_structure(frame):
    for ax, (temperature_K, data) in zip(axes_3d, records):
        ax.clear()
        cell = data["cell"]
        positions = wrapped_positions(data, frame)
        symbols = data["symbols"]
        for element in np.unique(symbols):
            mask = symbols == element
            ax.scatter(*positions[mask].T, s=18 if element == "O" else 34,
                       color=colors.get(str(element), "gray"), label=str(element), depthshade=True)
        draw_cell(ax, cell)
        ax.set_xlabel("x (Å)"); ax.set_ylabel("y (Å)"); ax.set_zlabel("z (Å)")
        field_z = data["trajectory_fields_z"][frame]
        ax.set_title(f"{temperature_K:g} K | step {frame + 1} | Ez={field_z:+.4f} V/Å")
        ax.legend(loc="upper right", fontsize=7)
    return []

animation = FuncAnimation(fig, update_structure, frames=frame_ids, interval=180, blit=False)
plt.close(fig)
display(HTML(animation.to_jshtml(fps=5)))
print("ASE trajectory files:")
for temperature_K, data in records:
    print(f"{temperature_K:g} K:", data["trajectory_path"])


## Interpretation

This remains a single-cell model demonstration, not a converged switching study. The ±0.36 V/Å field range is far outside the bundled response labels and may produce unstable or nonphysical extrapolations. The small periodic cell cannot represent realistic domain nucleation. Quantitative coercive fields or hysteresis require reference labels across both polarities and switching paths, plus larger cells and convergence checks.